In [52]:
import pandas as pd

#Variables
PICKUP, DROPOFF = "tpep_pickup_datetime", "tpep_dropoff_datetime"
COLUMNS = [PICKUP, DROPOFF, "trip_distance", "PULocationID", "DOLocationID"]
DISTANCE_THRESHOLD = 40

def read_columns(path):
    return pd.read_parquet(path, columns=COLUMNS)

# returns string for what data was and is currently and what ammount was dropped
def report(before, after):
    return f"Previous Dataset Size: {before:,} \nCurrent Dataset Size: {after:,} \nDropped: {before - after:,} lines \n{1-(after / before):.2%} Dropped \n{after / before:.2%} Preserved"

In [53]:
df = read_columns("OriginalData_DoNotDelete/yellow_tripdata_2026-Jan.parquet")
df.shape[0]

3724889

In [54]:
# validate duration anything greater than 0
df["duration_min"] = (df[DROPOFF] - df[PICKUP]).dt.total_seconds()/60
df_duration = df.loc[df["duration_min"] > 0].copy().reset_index(drop=True)
print(report(df.shape[0],df_duration.shape[0]))

Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,679,819 
Dropped: 45,070 lines 
1.21% Dropped 
98.79% Preserved


In [55]:
# filter data based on trip distance anything greater than 0 (not including 0) up to the threshold (currently 40 inclusive)
df_distance = df_duration.loc[df_duration["trip_distance"].between(0,DISTANCE_THRESHOLD, inclusive="right")].copy().reset_index(drop=True)
print(report(df_duration.shape[0], df_distance.shape[0]))

Previous Dataset Size: 3,679,819 
Current Dataset Size: 3,553,975 
Dropped: 125,844 lines 
3.42% Dropped 
96.58% Preserved


In [56]:
#Drop zone 264 and 265
df_zone = df_distance.loc[(df_distance["PULocationID"] < 264) & (df_distance["DOLocationID"] < 264)].copy().reset_index(drop=True)
print(report(df_distance.shape[0],df_zone.shape[0]))

Previous Dataset Size: 3,553,975 
Current Dataset Size: 3,531,522 
Dropped: 22,453 lines 
0.63% Dropped 
99.37% Preserved


In [57]:
df_clean = df_zone.copy()
print(report(df.shape[0],df_clean.shape[0]))

#save to new parquet file
path_to_data = "CleanedData/yellow_tripdata_2026-Jan_Cleaned-Dur-Dis-Zone264-265.parquet"
df_clean.to_parquet(path_to_data)
print(f"Saved to {path_to_data}")

Previous Dataset Size: 3,724,889 
Current Dataset Size: 3,531,522 
Dropped: 193,367 lines 
5.19% Dropped 
94.81% Preserved
Saved to CleanedData/yellow_tripdata_2026-Jan_Cleaned-Dur-Dis-Zone264-265.parquet
